In [1]:
import plotly, statsmodels, prophet, sklearn
print("plotly", plotly.__version__)
print("statsmodels", statsmodels.__version__)
print("prophet", prophet.__version__)
print("All libraries imported fine")


plotly 7.1.0
statsmodels 0.15.0
prophet 1.5.0
All libraries imported fine


# Air Quality Index (AQI) Analysis & Forecasting - Indian Cities
**Data source:** CPCB data via the Kaggle dataset "Air Quality Data in India (2015-2020)" (`city_day.csv`).
**Goal:** analyse pollution patterns, forecast future AQI with ARIMA and Prophet, and present insights through interactive dashboards.

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("Libraries loaded")

Libraries loaded


## 1. Load and inspect

In [3]:
df = pd.read_csv("city_day.csv")
print("Shape:", df.shape)
display(df.head())
print("Cities:", df["City"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Shape: (29531, 16)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


Cities: 26
Date range: 2015-01-01 to 2020-07-01


In [4]:
test = px.bar(x=["A", "B", "C"], y=[3, 7, 5], title="Plotly test chart")
test.show()

## 2. Data cleaning

In [5]:
cities = ["Delhi", "Mumbai", "Kolkata", "Bengaluru"]

aqi = df[df["City"].isin(cities)].copy()
aqi["Date"] = pd.to_datetime(aqi["Date"])
aqi = aqi.sort_values(["City", "Date"]).reset_index(drop=True)

print("Rows kept:", len(aqi))
print(aqi.groupby("City")["Date"].agg(["min", "max", "count"]))

Rows kept: 6841
                 min        max  count
City                                  
Bengaluru 2015-01-01 2020-07-01   2009
Delhi     2015-01-01 2020-07-01   2009
Kolkata   2018-04-10 2020-07-01    814
Mumbai    2015-01-01 2020-07-01   2009


In [6]:
missing = aqi.groupby("City").apply(lambda g: g.isna().mean() * 100).round(1)
print("Percent missing, per city and column:")
display(missing.T)

Percent missing, per city and column:


City,Bengaluru,Delhi,Kolkata,Mumbai
Date,0.0,0.0,0.0,0.0
PM2.5,7.3,0.1,6.8,61.0
PM10,17.9,3.8,6.8,62.0
NO,0.3,0.1,3.2,61.8
NO2,0.3,0.1,3.1,62.4
NOx,0.2,0.0,0.1,24.8
NH3,10.1,0.4,0.0,80.3
CO,0.5,0.0,0.0,1.2
SO2,0.3,5.5,7.6,60.7
O3,7.2,4.2,6.1,60.3


In [7]:
pollutants = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3"]
aqi = aqi[["City", "Date"] + pollutants + ["AQI"]].copy()

# Negative readings are impossible, so treat them as missing
num_cols = pollutants + ["AQI"]
for c in num_cols:
    aqi.loc[aqi[c] < 0, c] = np.nan

# The official AQI scale stops at 500
aqi["AQI"] = aqi["AQI"].clip(upper=500)

print("Columns kept:", list(aqi.columns))
print("Smallest AQI:", aqi["AQI"].min(), "| Largest AQI:", aqi["AQI"].max())

Columns kept: ['City', 'Date', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI']
Smallest AQI: 20.0 | Largest AQI: 500.0


In [8]:
parts = []
for city, g in aqi.groupby("City"):
    g = g.set_index("Date").sort_index()
    # time-based interpolation, but only across gaps of up to 14 days
    g[num_cols] = g[num_cols].interpolate(method="time", limit=14, limit_direction="both")
    g["City"] = city
    parts.append(g.reset_index())

aqi_clean = pd.concat(parts, ignore_index=True)

before = len(aqi_clean)
aqi_clean = aqi_clean.dropna(subset=["AQI"]).reset_index(drop=True)
print("Rows with no AQI that could not be filled (dropped):", before - len(aqi_clean))
print("Rows left:", len(aqi_clean))

Rows with no AQI that could not be filled (dropped): 1304
Rows left: 5537


In [9]:
def aqi_category(v):
    if v <= 50:   return "Good"
    if v <= 100:  return "Satisfactory"
    if v <= 200:  return "Moderate"
    if v <= 300:  return "Poor"
    if v <= 400:  return "Very Poor"
    return "Severe"

aqi_clean["AQI_Category"] = aqi_clean["AQI"].apply(aqi_category)
aqi_clean["Year"] = aqi_clean["Date"].dt.year
aqi_clean["Month"] = aqi_clean["Date"].dt.month

display(aqi_clean.groupby(["City", "AQI_Category"]).size().unstack(fill_value=0))

AQI_Category,Good,Moderate,Poor,Satisfactory,Severe,Very Poor
City,,,,,,
Bengaluru,119,633,37,1150,0,5
Delhi,23,521,542,164,239,520
Kolkata,119,166,119,300,13,66
Mumbai,26,294,35,445,0,1


In [10]:
print("Duplicate city-date rows:", aqi_clean.duplicated(subset=["City", "Date"]).sum())
print("Missing AQI left:", aqi_clean["AQI"].isna().sum())
print("Rows per city:")
print(aqi_clean["City"].value_counts())
print("Missing values per column:")
print(aqi_clean[num_cols].isna().sum())

aqi_clean.to_csv("aqi_cleaned.csv", index=False)
print("Saved aqi_cleaned.csv")

Duplicate city-date rows: 0
Missing AQI left: 0
Rows per city:
City
Delhi        2009
Bengaluru    1944
Mumbai        801
Kolkata       783
Name: count, dtype: int64
Missing values per column:
PM2.5      0
PM10     261
NO2        0
SO2       59
CO         0
O3        53
AQI        0
dtype: int64
Saved aqi_cleaned.csv


## 3. Exploratory data analysis

In [11]:
fig = px.box(aqi_clean, x="City", y="AQI", color="City",
             title="AQI distribution by city")
fig.show()

display(aqi_clean.groupby("City")["AQI"].describe().round(1))

,count,mean,std,min,25%,50%,75%,max
City,,,,,,,,
Bengaluru,1944.0,94.2,40.0,20.0,67.0,86.0,111.0,352.0
Delhi,2009.0,257.2,116.3,29.0,159.0,257.0,345.0,500.0
Kolkata,783.0,138.8,103.0,26.0,61.0,94.0,201.5,475.0
Mumbai,801.0,105.0,47.0,40.0,69.0,90.0,134.0,307.0


In [12]:
yearly = aqi_clean.groupby(["Year", "City"])["AQI"].mean().round(1).reset_index()

fig = px.line(yearly, x="Year", y="AQI", color="City", markers=True,
              title="Average AQI per year")
fig.update_xaxes(dtick=1)
fig.show()

In [13]:
monthly = aqi_clean.groupby(["Month", "City"])["AQI"].mean().round(1).reset_index()

fig = px.line(monthly, x="Month", y="AQI", color="City", markers=True,
              title="Average AQI by month of the year")
fig.update_xaxes(dtick=1)
fig.show()

pivot = aqi_clean.pivot_table(values="AQI", index="City", columns="Month", aggfunc="mean").round(0)
fig = px.imshow(pivot, text_auto=True, color_continuous_scale="YlOrRd",
                aspect="auto", title="Average AQI: City x Month")
fig.show()

In [14]:
corr = aqi_clean[num_cols].corr().round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Correlation between pollutants and AQI")
fig.show()

## 4. High-risk zones and peak periods

In [15]:
# "High risk" = days in Poor, Very Poor or Severe (AQI above 200)
aqi_clean["High_Risk"] = aqi_clean["AQI"] > 200
risk = (aqi_clean.groupby("City")["High_Risk"].mean() * 100).round(1).sort_values(ascending=False)
print("Percent of days in Poor / Very Poor / Severe:")
display(risk)

order = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
colors = {"Good": "#2e7d32", "Satisfactory": "#9ccc65", "Moderate": "#fdd835",
          "Poor": "#fb8c00", "Very Poor": "#e53935", "Severe": "#7b1fa2"}

share = (pd.crosstab(aqi_clean["City"], aqi_clean["AQI_Category"], normalize="index") * 100).round(1)
share = share.reindex(columns=[c for c in order if c in share.columns])

fig = px.bar(share, x=share.index, y=share.columns, barmode="stack",
             color_discrete_map=colors,
             labels={"value": "% of days", "x": "City", "variable": "Category"},
             title="Share of days in each AQI category")
fig.show()

Percent of days in Poor / Very Poor / Severe:


City
Delhi        64.8
Kolkata      25.3
Mumbai        4.5
Bengaluru     2.2
Name: High_Risk, dtype: float64

In [16]:
worst = monthly.loc[monthly.groupby("City")["AQI"].idxmax()][["City", "Month", "AQI"]]
print("Worst month for each city:")
display(worst.reset_index(drop=True))

# Delhi, October and November, year by year (stubble burning and Diwali season)
d = aqi_clean[(aqi_clean["City"] == "Delhi") & (aqi_clean["Month"].isin([10, 11]))].copy()
d["Day"] = d["Date"].dt.strftime("%m-%d")
fig = px.line(d, x="Day", y="AQI", color="Year",
              title="Delhi AQI, October to November, by year")
fig.update_xaxes(categoryorder="category ascending")
fig.show()

Worst month for each city:


,City,Month,AQI
0,Bengaluru,1,111.7
1,Delhi,11,389.0
2,Kolkata,1,288.9
3,Mumbai,12,167.7


## 5. Interactive dashboard

In [17]:
kpi = aqi_clean.groupby("City").agg(
    Days=("AQI", "count"),
    Avg_AQI=("AQI", "mean"),
    Max_AQI=("AQI", "max"),
    Avg_PM25=("PM2.5", "mean"),
    Pct_High_Risk=("High_Risk", lambda s: s.mean() * 100),
).round(1)
display(kpi)

,Days,Avg_AQI,Max_AQI,Avg_PM25,Pct_High_Risk
City,,,,,
Bengaluru,1944,94.2,352.0,36.2,2.2
Delhi,2009,257.2,500.0,117.1,64.8
Kolkata,783,138.8,475.0,63.0,25.3
Mumbai,801,105.0,307.0,35.1,4.5


In [18]:
from plotly.subplots import make_subplots

city_list = sorted(aqi_clean["City"].unique())
per_city = 5   # traces drawn for each city

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "xy"}, {"type": "domain"}], [{"type": "xy"}, {"type": "xy"}]],
    subplot_titles=("Daily AQI with 30-day average", "Days by AQI category",
                    "Average AQI by month", "Average pollutant levels"))

for i, city in enumerate(city_list):
    g = aqi_clean[aqi_clean["City"] == city].sort_values("Date")
    show = (i == 0)
    fig.add_trace(go.Scatter(x=g["Date"], y=g["AQI"], mode="lines", name="Daily AQI",
                             line=dict(color="lightgray", width=1), visible=show), row=1, col=1)
    fig.add_trace(go.Scatter(x=g["Date"], y=g["AQI"].rolling(30, min_periods=1).mean(),
                             mode="lines", name="30-day average",
                             line=dict(color="crimson", width=2), visible=show), row=1, col=1)
    counts = g["AQI_Category"].value_counts().reindex(order).dropna()
    fig.add_trace(go.Pie(labels=counts.index, values=counts.values, sort=False,
                         marker=dict(colors=[colors[c] for c in counts.index]),
                         showlegend=False, visible=show), row=1, col=2)
    m = g.groupby("Month")["AQI"].mean()
    fig.add_trace(go.Bar(x=m.index, y=m.values, marker_color="steelblue",
                         showlegend=False, visible=show), row=2, col=1)
    p = g[pollutants].mean()
    fig.add_trace(go.Bar(x=p.index, y=p.values, marker_color="darkorange",
                         showlegend=False, visible=show), row=2, col=2)

buttons = []
for i, city in enumerate(city_list):
    vis = [False] * (per_city * len(city_list))
    for k in range(per_city):
        vis[i * per_city + k] = True
    buttons.append(dict(label=city, method="update",
                        args=[{"visible": vis}, {"title": f"AQI dashboard: {city}"}]))

fig.update_layout(title=f"AQI dashboard: {city_list[0]}", height=700,
                  updatemenus=[dict(buttons=buttons, direction="down",
                                    x=0.0, y=1.18, xanchor="left")])
fig.show()

In [19]:
weekly_poll = (aqi_clean.set_index("Date").groupby("City")[pollutants]
               .resample("W").mean().reset_index())

fig = go.Figure()
for j, pol in enumerate(pollutants):
    for city in city_list:
        w = weekly_poll[weekly_poll["City"] == city]
        fig.add_trace(go.Scatter(x=w["Date"], y=w[pol], mode="lines", name=city,
                                 visible=(j == 0)))

buttons = []
for j, pol in enumerate(pollutants):
    vis = [False] * (len(pollutants) * len(city_list))
    for k in range(len(city_list)):
        vis[j * len(city_list) + k] = True
    buttons.append(dict(label=pol, method="update",
                        args=[{"visible": vis}, {"title": f"Weekly average {pol} by city"}]))

fig.update_layout(title=f"Weekly average {pollutants[0]} by city", height=500,
                  updatemenus=[dict(buttons=buttons, direction="down",
                                    x=0.0, y=1.15, xanchor="left")])
fig.show()

In [20]:
import calendar

avg = aqi_clean.groupby("City")["AQI"].mean().round(1).sort_values()
print("KEY INSIGHTS")
print(f"- Most polluted city: {avg.index[-1]} (average AQI {avg.iloc[-1]})")
print(f"- Cleanest city: {avg.index[0]} (average AQI {avg.iloc[0]})")
print(f"- Highest share of Poor / Very Poor / Severe days: {risk.index[0]} ({risk.iloc[0]}% of days)")
for _, r in worst.iterrows():
    print(f"- {r['City']}: worst month is {calendar.month_name[int(r['Month'])]} (average AQI {r['AQI']})")

KEY INSIGHTS
- Most polluted city: Delhi (average AQI 257.2)
- Cleanest city: Bengaluru (average AQI 94.2)
- Highest share of Poor / Very Poor / Severe days: Delhi (64.8% of days)
- Bengaluru: worst month is January (average AQI 111.7)
- Delhi: worst month is November (average AQI 389.0)
- Kolkata: worst month is January (average AQI 288.9)
- Mumbai: worst month is December (average AQI 167.7)


In [21]:
fig.write_html("aqi_dashboard.html")
print("Saved aqi_dashboard.html - double-click it to open in any browser")

Saved aqi_dashboard.html - double-click it to open in any browser


## 6. Forecasting preparation

In [22]:
weekly = (aqi_clean.set_index("Date").groupby("City")["AQI"]
          .resample("W").mean().reset_index())

print("Weekly points per city:")
print(weekly.groupby("City")["AQI"].count())
print("Missing weekly values:", weekly["AQI"].isna().sum())

fig = px.line(weekly, x="Date", y="AQI", color="City", title="Weekly average AQI")
fig.show()

Weekly points per city:
City
Bengaluru    279
Delhi        288
Kolkata      114
Mumbai       115
Name: AQI, dtype: int64
Missing weekly values: 3


## 7. Forecasting: ARIMA vs Prophet

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

test_weeks = 26   # hold back the last 26 weeks to test the models

splits = {}
for city in city_list:
    s = weekly[weekly["City"] == city].set_index("Date")["AQI"].asfreq("W-SUN").interpolate()
    splits[city] = (s.iloc[:-test_weeks], s.iloc[-test_weeks:])   # (train, test)

def metrics(actual, pred):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    return round(mae, 2), round(rmse, 2), round(mape, 2)

for city, (tr, te) in splits.items():
    print(f"{city}: train {len(tr)} weeks ({tr.index.min().date()} to {tr.index.max().date()}), "
          f"test {len(te)} weeks")

Bengaluru: train 253 weeks (2015-03-08 to 2020-01-05), test 26 weeks
Delhi: train 262 weeks (2015-01-04 to 2020-01-05), test 26 weeks
Kolkata: train 91 weeks (2018-04-15 to 2020-01-05), test 26 weeks
Mumbai: train 89 weeks (2018-04-29 to 2020-01-05), test 26 weeks


In [24]:
import itertools
from statsmodels.tsa.arima.model import ARIMA

def fit_arima(train):
    """Try a small grid of (p, d, q) settings and keep the one with the lowest AIC."""
    best = None
    for p, d, q in itertools.product(range(0, 4), [0, 1], range(0, 3)):
        try:
            model = ARIMA(train, order=(p, d, q)).fit()
            if best is None or model.aic < best[0]:
                best = (model.aic, (p, d, q), model)
        except Exception:
            continue
    return best

arima_pred, arima_order = {}, {}
for city, (tr, te) in splits.items():
    aic, order, model = fit_arima(tr)
    arima_order[city] = order
    arima_pred[city] = model.forecast(steps=len(te))
    print(f"{city}: best order {order}, test metrics (MAE, RMSE, MAPE) =",
          metrics(te, arima_pred[city]))

c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_

Bengaluru: best order (3, 1, 1), test metrics (MAE, RMSE, MAPE) = (17.23, np.float64(20.78), np.float64(26.06))
Delhi: best order (2, 1, 2), test metrics (MAE, RMSE, MAPE) = (238.41, np.float64(248.9), np.float64(174.68))


c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params


Kolkata: best order (3, 1, 1), test metrics (MAE, RMSE, MAPE) = (137.0, np.float64(154.71), np.float64(224.51))


c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
c:\Users\DELL\OneDrive\Desktop\SPARKIIT_PROJECTS\venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super(

Mumbai: best order (3, 1, 2), test metrics (MAE, RMSE, MAPE) = (111.87, np.float64(120.42), np.float64(157.25))


In [25]:
import logging
from prophet import Prophet
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

def fit_prophet(train, n_ahead):
    dfp = train.reset_index()
    dfp.columns = ["ds", "y"]
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(dfp)
    future = m.make_future_dataframe(periods=n_ahead, freq="W")
    return m, m.predict(future)

prophet_pred = {}
for city, (tr, te) in splits.items():
    m, fc = fit_prophet(tr, len(te))
    prophet_pred[city] = pd.Series(fc["yhat"].tail(len(te)).values, index=te.index)
    print(f"{city}: test metrics (MAE, RMSE, MAPE) =", metrics(te, prophet_pred[city]))

17:53:55 - cmdstanpy - INFO - Chain [1] start processing
17:53:55 - cmdstanpy - INFO - Chain [1] done processing
17:53:55 - cmdstanpy - INFO - Chain [1] start processing
17:53:56 - cmdstanpy - INFO - Chain [1] done processing


Bengaluru: test metrics (MAE, RMSE, MAPE) = (15.79, np.float64(19.41), np.float64(21.25))
Delhi: test metrics (MAE, RMSE, MAPE) = (55.55, np.float64(69.68), np.float64(44.24))


Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
17:53:56 - cmdstanpy - INFO - Chain [1] start processing
17:53:56 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
17:53:56 - cmdstanpy - INFO - Chain [1] start processing


Kolkata: test metrics (MAE, RMSE, MAPE) = (66.23, np.float64(68.95), np.float64(88.05))


17:53:56 - cmdstanpy - INFO - Chain [1] done processing


Mumbai: test metrics (MAE, RMSE, MAPE) = (17.48, np.float64(21.97), np.float64(19.7))


In [26]:
rows = []
for city, (tr, te) in splits.items():
    for name, pred in [("ARIMA", arima_pred[city]), ("Prophet", prophet_pred[city])]:
        mae, rmse, mape = metrics(te, pred)
        rows.append({"City": city, "Model": name, "MAE": mae, "RMSE": rmse, "MAPE_%": mape})

results = pd.DataFrame(rows)
display(results)

best_model = results.loc[results.groupby("City")["RMSE"].idxmin()].set_index("City")["Model"]
print("Best model per city (lowest RMSE):")
display(best_model.to_frame("Best model"))

fig = px.bar(results, x="City", y="RMSE", color="Model", barmode="group",
             title="Forecast error (RMSE) by city and model: lower is better")
fig.show()

,City,Model,MAE,RMSE,MAPE_%
0,Bengaluru,ARIMA,17.23,20.78,26.06
1,Bengaluru,Prophet,15.79,19.41,21.25
2,Delhi,ARIMA,238.41,248.90,174.68
3,Delhi,Prophet,55.55,69.68,44.24
4,Kolkata,ARIMA,137.00,154.71,224.51
5,Kolkata,Prophet,66.23,68.95,88.05
6,Mumbai,ARIMA,111.87,120.42,157.25
7,Mumbai,Prophet,17.48,21.97,19.70


Best model per city (lowest RMSE):


,Best model
City,
Bengaluru,Prophet
Delhi,Prophet
Kolkata,Prophet
Mumbai,Prophet


In [28]:
city = "Delhi"
tr, te = splits[city]
recent = tr.iloc[-52:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=recent.index, y=recent.values, name="Training data",
                         line=dict(color="lightgray")))
fig.add_trace(go.Scatter(x=te.index, y=te.values, name="Actual",
                         line=dict(color="black", width=2)))
fig.add_trace(go.Scatter(x=te.index, y=arima_pred[city].values, name="ARIMA",
                         line=dict(color="royalblue", dash="dash")))
fig.add_trace(go.Scatter(x=te.index, y=prophet_pred[city].values, name="Prophet",
                         line=dict(color="crimson", dash="dash")))
fig.update_layout(title=f"Predicted vs actual AQI: {city}", height=500)
fig.show()

print("ARIMA orders chosen:", arima_order)
print("Delhi ARIMA forecast, first 5 weeks:", arima_pred[city].head().round(0).tolist())
print("Delhi ARIMA forecast, last 5 weeks: ", arima_pred[city].tail().round(0).tolist())
print("Delhi actual, first 5 weeks:        ", te.head().round(0).tolist())

ARIMA orders chosen: {'Bengaluru': (3, 1, 1), 'Delhi': (2, 1, 2), 'Kolkata': (3, 1, 1), 'Mumbai': (3, 1, 2)}
Delhi ARIMA forecast, first 5 weeks: [412.0, 399.0, 411.0, 422.0, 416.0]
Delhi ARIMA forecast, last 5 weeks:  [413.0, 413.0, 413.0, 413.0, 413.0]
Delhi actual, first 5 weeks:         [293.0, 294.0, 292.0, 254.0, 312.0]


### Model comparison findings
- **Delhi:** Prophet performed better (RMSE 69.68 versus 248.90 for ARIMA).
- **Mumbai:** Prophet performed better (RMSE 21.97 versus 120.42 for ARIMA).
- **Kolkata:** Prophet performed better (RMSE 68.95 versus 154.71 for ARIMA).
- **Bengaluru:** Prophet performed better (RMSE 19.41 versus 20.78 for ARIMA), but the gap is small here.
- Prophet had the lower error in every city, so it was chosen for all four forecasts.
- The gap is largest in Delhi, Mumbai and Kolkata, where AQI swings strongly with the seasons. The training data ends in the winter peak, and ARIMA has no seasonal component, so its forecast stays flat at the winter level (about 413 for Delhi) while the real AQI fell. Prophet learns the yearly cycle and followed the drop. Bengaluru's AQI is flatter, so both models did similarly.
- The test period (January to July 2020) also includes the COVID lockdown, when AQI fell sharply. This raises the error for both models.